In [ ]:
import pandas as pd
import minio
import clickhouse_connect
import os
from datetime import datetime

pd.set_option('display.max_columns', None)

In [ ]:
vars = !cat .env
for var in vars:
    if var != "":
        key, val = var.split("=")
        os.environ[key] = val

print(os.environ.get("MINIO_AP"))

In [ ]:
# MinIo connection
mio = minio.Minio(
    endpoint=os.environ.get("MINIO_AP"),
    access_key=os.environ.get("MINIO_ACCESS_KEY"),
    secret_key=os.environ.get("MINIO_SECRET_KEY"),
    # Allos HTTP connetion
    secure=False
)

# Clickhouse connection
ch = clickhouse_connect.create_client(
    host=os.environ.get("CH_AP"),
    username=os.environ.get("CH_USERNAME"),
    password=os.environ.get("CH_PASSWORD"),
    database=os.environ.get("CH_DATABASE"),
    port=os.environ.get("CH_PORT"),
)

In [ ]:
# Clickhouse tables
ch.query("""
  CREATE DATABASE IF NOT EXISTS ingestion;
""")

In [ ]:
# Table for finn_homes

# Table for finn_lettings
ch.query("""
    CREATE OR REPLACE TABLE ingestion.finn_lettings (
        type                                String
        id                                  String
        main_search_key                     String
        heading                             String
        location                            String
        flags                               Array(String)
        timestamp                           Int64
        canonical_url                       String
        extras                              Array(String)
        organisation_name                   String
        local_area_name                     String
        number_of_bedrooms                  Int64
        property_type_description           String
        viewing_times                       Array(Int64)
        ad_type                             Int64
        furnished_state                     String
        ad_id                               Int64
        suggested_price                     Int64
        suggested_price_currency_code       String
        total_price                         Int64
        total_price_currency_code           String
        shared_cost_price                   Int64
        shared_cost_price_currency_code     String
        area_from                           Float64
        area_to                             Float64
        area_range_unit                     String
        plot_area                           String
        area_plot_unit                      String
        longitue                           float64
        latitude                           float64
        is_private_ad                         bool
        ingestion_ts DateTime64 DEFAULT current_timestamp()
    ) ENGINE = MergeTree()
    ORDER BY timestamp;
""")


In [ ]:
# Table for finn_homes
ch.query("""
    CREATE TABLE IF NOT EXISTS ingestion.finn_homes (
        type String,
        id String,
        main_search_key String,
        heading String,
        location String,
        flags String,
        timestamp String,
        canonical_url String,
        organisation_name String,
        local_area_name String,
        number_of_bedrooms String,
        owner_type_description String,
        property_type_description String,
        viewing_times String,
        ad_type String,
        ad_id String,
        suggested_price String,
        suggested_price_currency_code String,
        total_price String,
        total_price_currency_code String,
        shared_cost_price String,
        shared_cost_price_currency_code String,
        price_range_from String,
        price_range_to String,
        price_range_currency_code String,
        price_total_from String,
        price_total_to String,
        price_total_currency_code String,
        area_from String,
        area_to String,
        area_range_unit String,
        plot_area String,
        area_plot_unit String,
        longitude String,
        furnished_state String,
        latitude String,
        ingestion_ts DateTime64 DEFAULT current_timestamp()
    ) ENGINE = MergeTree()
    ORDER BY timestamp;
""")


In [ ]:
ch.query("drop table ingestion.finn_lettings")

In [ ]:

# Table for finn_lettings
ch.query("""
    CREATE TABLE IF NOT EXISTS ingestion.finn_lettings (
        type                                String,
        id                                  String,
        main_search_key                     String,
        heading                             String,
        location                            String,
        flags                               String,
        timestamp                           String,
        canonical_url                       String,
        extras                              String,
        organisation_name                   String,
        local_area_name                     String,
        number_of_bedrooms                  String,
        property_type_description           String,
        viewing_times                       String,
        ad_type                             String,
        furnished_state                     String,
        ad_id                               String,
        suggested_price                     String,
        suggested_price_currency_code       String,
        total_price                         String,
        total_price_currency_code           String,
        shared_cost_price                   String,
        shared_cost_price_currency_code     String,
        area_from                           String,
        area_to                             String,
        area_range_unit                     String,
        plot_area                           String,
        area_plot_unit                      String,
        longitude                           String,
        latitude                            String,
        is_private_ad                       String,
        ingestion_ts DateTime64 DEFAULT current_timestamp()
    ) ENGINE = MergeTree()
    ORDER BY timestamp;
""")


In [ ]:
# Table for ingesting raw jsin for later processing
ch.query("""
  CREATE TABLE IF NOT EXISTS ingestion.json (
    ingestion_ts DateTime64 DEFAULT current_timestamp(),
    json String,
    object_name String
  ) ENGINE = MergeTree() 
  ORDER BY ingestion_ts;
""")

In [ ]:
# Retrieving desired data for ingestion
lettings = [
    *mio.list_objects("ingestion", 
    prefix="finn_ingestion__SEARCH_ID_REALESTATE_LETTINGS")
]

homes = [
    *mio.list_objects("ingestion",
    prefix="finn_ingestion__SEARCH_ID_REALESTATE_HOMES")
]

all_objects = lettings + homes

In [ ]:
homes_object = mio.get_object("ingestion", home.object_name).json()
homes_df = pd.DataFrame(homes_object["docs"])

In [ ]:
homes_df["area_plot"]

In [ ]:
# Ingest home ads
for idx, home in enumerate(homes):
    print(f"ingesting {idx}/{len(homes)}")

    homes_object = mio.get_object("ingestion", home.object_name).json()
    homes_df = pd.DataFrame(homes_object["docs"])
 
    # Data cleaning
    homes_df = homes_df.drop(
        columns=[
            "image", 
            "styling", 
            "image_urls", 
            "logo", 
            "area",
            "labels",
            "bedrooms_range",
            "extras"
        ], 
        errors="ignore"
    )
    homes_df["suggested_price"] = homes_df["price_suggestion"].apply(lambda x: x["amount"] if isinstance(x, dict) and "amount" in x else None)
    homes_df["suggested_price_currency_code"] = homes_df["price_suggestion"].apply(lambda x: x["currency_code"] if isinstance(x, dict) and "currency_code" in x else None )
    homes_df = homes_df.drop(columns=["price_suggestion"])

    homes_df["total_price"] = homes_df["price_total"].apply(lambda x: x["amount"]  if isinstance(x, dict) and "amount" in x else None)
    homes_df["total_price_currency_code"] = homes_df["price_total"].apply(lambda x: x["currency_code"] if isinstance(x, dict) and "currency_code" in x else None)
    homes_df = homes_df.drop(columns=["price_total"])

    homes_df["shared_cost_price"] = homes_df["price_shared_cost"].apply(lambda x: x["amount"] if isinstance(x, dict) and "amount" in x else None)
    homes_df["shared_cost_price_currency_code"] = homes_df["price_shared_cost"].apply(lambda x: x["currency_code"] if isinstance(x, dict) and "currenc_code" in x else None)
    homes_df = homes_df.drop(columns=["price_shared_cost"])



    homes_df["price_range_from"] = homes_df["price_range_suggestion"].apply(lambda x: x["amount_from"] if isinstance(x, dict) and "amount_from" in x else None)
    homes_df["price_range_to"] = homes_df["price_range_suggestion"].apply(lambda x: x["amount_to"] if isinstance(x, dict) and "amount_to" in x else None)
    homes_df["price_range_currency_code"] = homes_df["price_range_suggestion"].apply(lambda x: x["currency_code"] if isinstance(x, dict) and "currency_code" in x else None)
    homes_df = homes_df.drop(columns=["price_range_suggestion"])

    homes_df["price_total_from"] = homes_df["price_range_total"].apply(lambda x: x["amount_from"] if isinstance(x, dict) and "amount_from" in x else None)
    homes_df["price_total_to"] = homes_df["price_range_total"].apply(lambda x: x["amount_to"] if isinstance(x, dict) and "amount_to" in x else None)
    homes_df["price_total_currency_code"] = homes_df["price_range_total"].apply(lambda x: x["currency_code"] if isinstance(x, dict) and "currency_code" in x else None)
    homes_df = homes_df.drop(columns=["price_range_total"])


    homes_df["area_from"] = homes_df["area_range"].apply(lambda x: x["size_from"] if isinstance(x, dict) and  "size_from" in x else None)
    homes_df["area_to"] = homes_df["area_range"].apply(lambda x: x["size_to"] if isinstance(x, dict) and  "size_to" in x else None)
    homes_df["area_range_unit"] = homes_df["area_range"].apply(lambda x: x["unit"] if isinstance(x, dict) and  "unit" in x else None)
    homes_df = homes_df.drop(columns=["area_range"])

    homes_df["plot_area"] = homes_df["area_plot"].apply(lambda x: x["size"] if isinstance(x, dict) and  "size" in x else None)
    homes_df["area_plot_unit"] = homes_df["area_plot"].apply(lambda x: x["unit"] if isinstance(x, dict) and  "unit" in x else None)
    homes_df = homes_df.drop(columns=["area_plot"])

    homes_df["longitude"] = homes_df["coordinates"].apply(lambda x: x["lon"] if isinstance(x, dict) and  "lon" in x else None)
    homes_df["latitude"] = homes_df["coordinates"].apply(lambda x: x["lat"] if isinstance(x, dict) and "lat" in x else None)
    homes_df = homes_df.drop(columns=["coordinates"])


    for col in homes_df.columns:
        homes_df[col] = homes_df[col].astype("string")

    homes_df = homes_df.fillna("")

    # Inserting
    ch.insert_df(
        table="finn_homes", 
        df=homes_df, 
        database="ingestion", 
    )

    print("successfully inserted into clickhouse!")


In [ ]:
# Ingesting lettings
for idx, letting in enumerate(lettings):
    print(f"ingesting {idx}/{len(lettings)}")
    
    lettings_object = mio.get_object("ingestion", letting.object_name).json()

    lettings_df = pd.DataFrame(lettings_object["docs"])

    # Data cleaning
    lettings_df = lettings_df.drop(
        columns=[
            "image", 
            "styling", 
            "image_urls", 
            "logo", 
            "area",
            "labels"
        ], 
        errors="ignore"
    )
    
    lettings_df["suggested_price"] = lettings_df["price_suggestion"].apply(lambda x: x["amount"] if isinstance(x, dict) and "amount" in x else None)
    lettings_df["suggested_price_currency_code"] = lettings_df["price_suggestion"].apply(lambda x: x["currency_code"] if isinstance(x, dict) and "currency_code" in x else None )
    lettings_df = lettings_df.drop(columns=["price_suggestion"])

    lettings_df["total_price"] = lettings_df["price_total"].apply(lambda x: x["amount"]  if isinstance(x, dict) and "amount" in x else None)
    lettings_df["total_price_currency_code"] = lettings_df["price_total"].apply(lambda x: x["currency_code"] if isinstance(x, dict) and "currency_code" in x else None)
    lettings_df = lettings_df.drop(columns=["price_total"])


    lettings_df["shared_cost_price"] = lettings_df["price_shared_cost"].apply(lambda x: x["amount"] if isinstance(x, dict) and "amount" in x else None)
    lettings_df["shared_cost_price_currency_code"] = lettings_df["price_shared_cost"].apply(lambda x: x["currency_code"] if isinstance(x, dict) and "currenc_code" in x else None)
    lettings_df = lettings_df.drop(columns=["price_shared_cost"])

    lettings_df["area_from"] = lettings_df["area_range"].apply(lambda x: x["size_from"] if isinstance(x, dict) and  "size_from" in x else None)
    lettings_df["area_to"] = lettings_df["area_range"].apply(lambda x: x["size_to"] if isinstance(x, dict) and  "size_to" in x else None)
    lettings_df["area_range_unit"] = lettings_df["area_range"].apply(lambda x: x["unit"] if isinstance(x, dict) and  "unit" in x else None)
    lettings_df = lettings_df.drop(columns=["area_range"])

    lettings_df["plot_area"] = lettings_df["area_plot"].apply(lambda x: x["size"] if isinstance(x, dict) and  "size_from" in x else None)
    lettings_df["area_plot_unit"] = lettings_df["area_plot"].apply(lambda x: x["unit"] if isinstance(x, dict) and  "unit" in x else None)
    lettings_df = lettings_df.drop(columns=["area_plot"])

    lettings_df["longitude"] = lettings_df["coordinates"].apply(lambda x: x["lon"] if isinstance(x, dict) and  "lon" in x else None)
    lettings_df["latitude"] = lettings_df["coordinates"].apply(lambda x: x["lat"] if isinstance(x, dict) and "lat" in x else None)
    lettings_df = lettings_df.drop(columns=["coordinates"])


    lettings_df["is_private_ad"] = lettings_df["flags"].apply(lambda x: True if isinstance(x, list) and "private" in x else False)

    for col in lettings_df.columns:
        lettings_df[col] = lettings_df[col].astype("string")

    lettings_df = lettings_df.fillna("")

    # Inserting
    ch.insert_df(
        table="finn_lettings", 
        df=lettings_df, 
        database="ingestion", 
    )

    print("successfully inserted into clickhouse!")

In [ ]:
lettings_df = pd.DataFrame(lettings_object["docs"])

In [ ]:
lettings_df["area_range"]

In [ ]:
# Ingesting data

obj_total = len(all_objects)
for num, obj_entry in enumerate(all_objects):
    obj_name = obj_entry.object_name

    print(f"ingesting {num}/{obj_total}: {obj_name}")
    
    obj = mio.get_object(
        bucket_name="ingestion", 
        object_name=obj_name,
    )

    res = ch.insert(
        "json", 
        data=[[obj.data, obj_name]],
        column_names=["json", "object_name"],
        database="ingestion"
    )

    print(f"wrote: {res.written_bytes()} bytes")
    

In [ ]:
ch.insert(
    "json", 
    data=[[obj.data]],
    column_names="json",
    database="ingestion"
)